# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']
I. J. M. Crossfield  ->  I. J. M. Crossfield  |  ['I. J. M. Crossfield']
E. Schinnerer  ->  E. Schinnerer  |  ['E. Schinnerer']
J. Li  ->  J. Li  |  ['J. Li']
H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']
J. Müller-Horn  ->  J. Müller-Horn  |  ['J. Müller-Horn']
K. El-Badry  ->  K. El-Badry  |  ['K. El-Badry']
B. Pennell  ->  B. Pennell  |  ['B. Pennell']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']


H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']
Y.-S. Ting  ->  Y.-S. Ting  |  ['Y.-S. Ting']
J. Li  ->  J. Li  |  ['J. Li']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Arxiv has 137 new papers today
          8 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/8 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.28209


extracting tarball to tmp_2611.28209... done.


A. d. Graaff  ->  A. D. Graaff  |  ['A. D. Graaff']


Retrieving document from  https://arxiv.org/e-print/2611.28818
extracting tarball to tmp_2611.28818... done.
Retrieving document from  https://arxiv.org/e-print/2611.29560


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.28818/sample701.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'Table_4_updated_28th_sept' from 'tmp_2611.28818/Table_4_updated_28th_sept.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'table_3_updated_28th_sept' from 'tmp_2611.28818/table_3_updated_28th_sept.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/o

... (8 outputs left out)

 done.
Retrieving document from  https://arxiv.org/e-print/2611.31231
extracting tarball to tmp_2611.31231...

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.31231/blank.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.31231/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


/tmp/ipykernel_2898/2822249172.py:52: LatexWarning: 2611.31231 did not run properly
list index out of range
  warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.28209-b31b1b.svg)](https://arxiv.org/abs/2611.28209) | **Quasi-hostless LRDs with intrinsically red, faint UV continua**  |
|| C. Witten, et al. -- incl., <mark>A. d. Graaff</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| *15 pages, 8 figures, 1 table. Submitted to A&A. Comments are welcome*|
|**Abstract**|            High-redshift objects characterised by very weak UV relative to their strong, red rest-optical continua, with broad Balmer lines, appear to represent little red dots (LRDs) in their purest form: sources dominated by emission from their central engine (CE), sometimes dubbed black hole stars (BH*s). Previous work has selected BH*-dominated objects, where a BH* template dominates the rest-optical emission; this remains agnostic to the rest-UV. We therefore select low FUV-to-optical continuum ratio ($L_{\rm 1500}/L_{5100}<0.03$) objects resembling MoM-BH* and The Cliff, from a spectroscopic sample of 132 LRDs at $2\lesssim z\lesssim9$. This identifies three LRDs that we newly classify as BH*-like objects, more than doubling the existing sample: two at $z>6$ and one at $z\sim3$. The UV slope reddens as the NUV- ($L_{\rm 3500}$) and FUV-to-optical ratios fall, with $\beta_{\rm UV}$ rising from $-1.8$ at $L_{\rm 1500}/L_{5100}\sim 0.5$ to $-0.5$ at $L_{\rm 1500}/L_{5100}<0.03$, beyond values typically seen at these redshifts. As the UV-to-optical ratio weakens, gas-enshrouded BH emission appears to dominate, leading to the quasi-hostless regime: a weak, red UV continuum emerges alongside extreme Balmer decrements and weak forbidden line emission. This reddening could be reproduced by a simple model of a blackbody and hydrogen nebular emission leaking through a high column density screen of neutral hydrogen in the $n=2$ level ($N_{\rm H,\,n=2}\approx10^{17}\,{\rm cm^{-2}}$), combined with a relatively weak host galaxy, although alternative CE scenarios may also reproduce this reddening. The host galaxy's growing FUV contribution in this model results in a testable UV morphological evolution with wavelength. This work identifies a family of quasi-hostless LRDs that share a common, intrinsically red, rest-UV continuum originating from their central engine.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.29560-b31b1b.svg)](https://arxiv.org/abs/2611.29560) | **Discovery of Two Hyperluminous Infrared Galaxies at $z\sim6$**  |
|| S. Jin, et al. -- incl., <mark>E. Schinnerer</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| *Submitted to A&A*|
|**Abstract**|            Hyperluminous infrared galaxies (HyLIRGs) at cosmic dawn are key to understanding the formation of the first massive galaxies. We report the spectroscopic confirmation of two unlensed HyLIRGs at $z\sim6$ in the COSMOS field, AzTECC63 and S2COS-84, originally selected from single-dish AzTEC and SCUBA-2 surveys. Using ALMA and NOEMA observations, we robustly detected CO and [CII] lines in both galaxies and unambiguously confirmed their redshifts to be $z=6.092$ and $z=5.90$, respectively. Panchromatic SED fitting reveals dust-obscured star formation rates of $\approx1000$ and $\approx2300\,M_\odot\,{\rm yr}^{-1}$, ranking them among the most intensely star-forming systems known at $z \gtrsim 6$. High-resolution ALMA observations reveal an extremely compact morphology for S2COS-84, with a dust-continuum effective radius of $\approx0.3$ kpc, implying a star-formation rate surface density above the Eddington limit. The far-infrared SEDs of both galaxies, together with the compact size of S2COS-84, indicate warm dust ($T_{\rm dust} \approx 80$ K) that is likely optically thick at far-infrared wavelengths. Both galaxies exhibit an extreme [CII] deficit, with $L_{\rm [C\,II]}/L_{\rm IR} \approx 10^{-3.7}$, an order of magnitude below that of normal star-forming galaxies. Their molecular gas masses, derived consistently from both [C\,{\sc ii}] and dust, together with their IR-derived SFRs imply short gas depletion times of $\approx$40--60 Myr, making them plausible progenitors of the earliest massive quiescent galaxies found at $z\sim5$.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.29702-b31b1b.svg)](https://arxiv.org/abs/2611.29702) | **Unresolved binaries from Gaia XP to SPHEREx**  |
|| <mark>J. Li</mark>, et al. -- incl., <mark>H.-W. Rix</mark>, <mark>J. Müller-Horn</mark>, <mark>K. El-Badry</mark>, <mark>B. Pennell</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| *24 pages, 16 figures, 11 tables*|
|**Abstract**|            Unresolved companions alter the luminosity and spectra of a star, but a colour-magnitude diagram (CMD) reveals them only when the system is over-luminous, and a Gaia photocentre orbit constrains only a combination of the mass and flux ratios. We fit the absolute Gaia BP/RP (XP) and SPHEREx spectra (XPS) of about 71,000 low-mass stars in the Gaia Catalogue of Nearby Stars with the J-CaPS XPS emulator, which predicts single-star and coeval-binary spectra on an absolute flux scale. First, more than half of known binaries lie within the main-sequence (MS) band of the CMD, because the companion is faint or lifts a metal-poor primary onto the single-star ridge. XPS recovers 86 per cent of the binaries that a CMD cut finds and 44 per cent of those it misses. Second, a small photocentre motion can come from a faint companion or a luminous one of nearly equal mass. XPS distinguishes them and recovers the mass ratios of double-lined binaries. Among substellar candidates with radial-velocity follow-up, it ranks all 11 binaries above all 16 confirmed brown-dwarf or planet hosts. The public package spbin provides this check before follow-up. Third, where XPS detects no companion light, we place an upper limit on the light of an MS companion. Among Gaia DR3 astrometric orbits with clean spectral fits, 2841 require more companion mass than XPS allows for a single MS star. Within 200 pc, their companion masses cluster at those of typical white dwarfs, and nearly circular orbits support this interpretation. Of 24 companions that joint radial-velocity and astrometric orbits show to be faint, XPS recovers 22, compared with 14 selected by the absence of a colour excess. Applied to forthcoming Gaia DR4 orbital solutions, XPS can help distinguish luminous binaries from substellar companions and identify candidate compact companions whose dynamical masses exceed the limits allowed by their light.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.30393-b31b1b.svg)](https://arxiv.org/abs/2611.30393) | **Dynamical calibration of a metallicity-dependent stellar mass--luminosity relation from Gaia wide binaries**  |
|| Y.-T. Wang, et al. -- incl., <mark>H.-W. Rix</mark>, <mark>Y.-S. Ting</mark>, <mark>J. Li</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| *Submitted to A&A*|
|**Abstract**|            The mass of a main-sequence star is usually estimated from its luminosity through a mass--luminosity relation (MLR). This relation depends on metallicity, and its empirical calibration relies on few stars with measured masses. We calibrate the MLR of single main-sequence stars in the Gaia $G$ band as a function of metallicity with $10\,559$ wide binaries. The projected relative speed and separation of a wide binary constrain its total mass statistically. We describe the stellar mass as a monotone function of absolute magnitude and metallicity and fit this function to all pairs. The metallicities come from Gaia low-resolution (XP) spectra and are calibrated by requiring that both stars of a pair have the same metallicity. The likelihood includes the measurement noise, the selection of the sample and pairs with an undetected close companion. The data constrain the MLR over $3.7\lesssim M_{\mathrm{G}} \lesssim11.8$\,mag and $-0.5\lesssim\ \mathrm{[M/H]} \lesssim0.4$, with a median statistical uncertainty of $2.5\%$ in mass. Tests of the model assumptions indicate an additional systematic uncertainty of up to $5\%$. At solar metallicity the MLR agrees with the PARSEC models to within $5\%$. At the faint end, the MIST, BaSTI, Dartmouth and BHAC15 models predict masses 8 to $22\%$ below our MLR. Published masses from orbits, eclipsing binaries and asteroseismology lie within about $10\%$ of our MLR for $M_{\mathrm{G}}<10.5$ mag. For fainter stars they are about $17\%$ lower. The origin of the difference at the faint end is not yet understood. Because a shift of $0.1$ dex in metallicity changes the mass by up to $5\%$, MLRs should be compared on a common metallicity scale.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.28818-b31b1b.svg)](https://arxiv.org/abs/2611.28818) | **Reassessment of Planetary Parameters and Statistical Validation of 187 Kepler Habitable-Zone and Near Habitable Confirmed and Candidate Planets**  |
|| M. A. A. Zaman, <mark>I. J. M. Crossfield</mark>, E. M. Esmer, J. S. Pero |
|*Appeared on*| *2026-11-24*|
|*Comments*| *Accepted to AJ on 13th November 2026*|
|**Abstract**|            Confirmed and candidate planets in the Habitable or near-Habitable Zones (HZ) orbiting Sun-like stars are crucial not only for the search for life beyond the Solar System, but also for understanding the architecture and formation of other solar systems and for conducting demographic studies. Current analyses of such planetary systems are limited both by the small sample size revealed by Kepler as well as by uncertainties in system parameters. Although a larger sample of HZ and near-HZ planets must await new observations from wide-field, space-based surveys, follow-up analyses and observations with existing facilities can improve the precision on stellar and planetary parameters, enable more accurate planetary classification, and a more reliable assessment of whether candidate systems are true planets or merely astrophysical false positives. Here, we present a homogeneous transit light curve analysis of 187 HZ and near-HZ systems discovered by Kepler. Our analysis incorporates the latest Gaia DR3-derived stellar parameters, and we also present a statistical validation effort for all targets. The result is a uniform set of planetary radii, instellation fluxes, and orbital ephemerides suitable for prioritizing and scheduling follow-up observations to improve our understanding of planets in this regime. We show that longer-period (>300days) systems would particularly benefit from transit follow-up: just a single transit from facilities such as the 6m James Webb or the planned 3m Lazuli space telescopes could provide ~2 times tighter precision on planet radii than four years of Kepler photometry, thereby dramatically shrinking relevant parameter uncertainties and allowing more confident planet/false-positive assessments.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.30054-b31b1b.svg)](https://arxiv.org/abs/2611.30054) | **Environmental dependence of the properties of H$α$ emitters in the extremely massive galaxy protoclusters BOSS1244 and BOSS1542 at $z = 2.24$**  |
|| D. Shi, et al. -- incl., <mark>Y. Zhang</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| *18 pages, 8 figures, 6 tables, accepted for publication in A&A*|
|**Abstract**|            Protoclusters, the progenitors of galaxy clusters, provide ideal laboratories for studying how high-density environments influence galaxy formation and evolution at early cosmic epochs. It remains unclear whether environmental effects on galaxy properties over scales of $>30$ cMpc are already in place at $z > 2$ and how these effects depend on the evolutionary stage of large-scale structures. We investigate the physical properties of H$\alpha$ emitters (HAEs) in two extremely overdense protoclusters, BOSS1244 and BOSS1542, at $z = 2.24$, and explore the environmental dependence of galaxy properties at this redshift. In BOSS1244, we find a clear color-density and stellar mass-density relation. HAEs in higher-density regions are significantly redder in $z_{\rm s}-K_{\rm s}$ and more massive than those in lower-density regions. In contrast, BOSS1542 shows weak or no such correlations. Red HAEs $(z_{\rm s}-K_{\rm s}>2.5)$ in both fields are preferentially located in the densest cores, have high stellar masses $(>10^{11}\,M_{\odot})$, high SFRs, and are $4-7$ times more abundant than in general fields. The irregular morphologies observed in some red HAEs are consistent with ongoing mass assembly and may be associated with dust-obscured star formation or galaxy interactions. Both protoclusters broadly follow the star-forming main sequence, but BOSS1542 exhibits a steeper slope, indicating more active star formation in massive galaxies. Our results demonstrate that environmental effects on galaxy properties operate over scales of $\sim39$\,cMpc at $z=2.24$, but their manifestation is strongly dependent on the evolutionary state of the large-scale structure. BOSS1244 appears more evolved, with ongoing galaxy merging and possible quenching, while BOSS1542 remains in an early, filamentary, gas-rich phase.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.30504-b31b1b.svg)](https://arxiv.org/abs/2611.30504) | **A Log-Normal Timing Likelihood for Direction Reconstruction of Ultra-High-Energy Gamma-Ray Air Showers**  |
|| J.-Y. Shen, et al. -- incl., <mark>Y. Zhang</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| **|
|**Abstract**|            Ultra-high-energy $\gamma$-ray observations provide a direct probe of Galactic PeVatrons. Extending measurements to multi-PeV energies requires characterizing extensive air showers over large lateral distances, where the arrival-time distribution of secondary particles becomes broad and strongly asymmetric. Conventional direction reconstruction methods approximate the shower front as a deterministic surface and minimize a $\chi^2$ function of timing residuals. Because this implicitly assumes symmetric Gaussian errors, far-axis timing information is poorly utilized. \textsc{CORSIKA} simulations show that the arrival-time distribution of $\gamma$-induced secondary particles in the shower plane at a given lateral distance is positively skewed and closely follows a log-normal function. Using this property, we develop a likelihood-based direction reconstruction method that directly models the empirical arrival-time distribution and its radial dependence while accounting for multiple particles triggering the same detector unit. Applied to simulated $\gamma$-induced air showers, the log-normal likelihood consistently improves angular resolution compared with conical and parabolic wavefront fits, with the largest relative gains for high-energy events spanning wide lateral extents. We also show that independent readout of individual detector units yields better angular resolution than merging their signals into a single station-level measurement.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.31231-b31b1b.svg)](https://arxiv.org/abs/2611.31231) | **Euclid Data Release 1 (DR1). Learning machine learning: Investigating the complementarity and biases of strong gravitational lens finding networks on Euclid data**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-24*|
|*Comments*| **|
|**Abstract**|            The Euclid Data Release 1 (DR1) strong lens search presents ECLIPSE, the largest catalogue of strong gravitational lens candidates to date, with >15 000 systems identified across 1900 deg2 of sky. It was constructed using five neural networks applied to approximately 68 million galaxies, followed by citizen science and expert visual inspection. In this work, we compare how the independently developed networks use image information to identify their selection of strong lens candidates. The search combines convolutional neural networks and a vision transformer, using the same pool of data but with different image processing and training choices. We investigate why these networks find different systems and what that means for the completeness of our catalogue. We evaluate all networks on a 50 deg2 calibration set and their application to the full DR1 sample. We find that the networks score candidates very differently, and that combining them recovers over 500 lens candidates within our calibration set at an inspection budget of 2000 objects. On the full DR1 sample, we show that individual networks have distinct biases: some networks favour systems with a larger Einstein radius while others are more sensitive to compact configurations. We also investigate network preferences in terms of lens light profile, mass distribution geometry, photometric redshift, and magnitudes. These findings provide a detailed analysis of how neural network selection biases have shaped the Euclid DR1 ECLIPSE catalogue. The results inform how networks should be combined in future Euclid data releases and upcoming wide-field surveys such as LSST or Roman to maximise both completeness and diversity of the recovered lens population.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.28209/./figures/3micron_vs_UV_slopes_vertical.png', 'tmp_2611.28209/./figures/stacked_spectra.png', 'tmp_2611.28209/./figures/emission_lines.png']
copying  tmp_2611.28209/./figures/3micron_vs_UV_slopes_vertical.png to _build/html/
copying  tmp_2611.28209/./figures/stacked_spectra.png to _build/html/
copying  tmp_2611.28209/./figures/emission_lines.png to _build/html/
exported in  _build/html/2611.28209.md
    + _build/html/tmp_2611.28209/./figures/3micron_vs_UV_slopes_vertical.png
    + _build/html/tmp_2611.28209/./figures/stacked_spectra.png
    + _build/html/tmp_2611.28209/./figures/emission_lines.png
found figures ['tmp_2611.29560/./figure1.png', 'tmp_2611.29560/./figure2.png', 'tmp_2611.29560/./LCII-LIR.png']
copying  tmp_2611.29560/./figure1.png to _build/html/
copying  tmp_2611.29560/./figure2.png to _build/html/
copying  tmp_2611.29560/./LCII-LIR.png to _build/html/
exported in  _build/html/2611.29560.md
    + _build/html/tmp_2611.29560/./figure1.png
   

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Lya}{Ly\rm{\alpha}}$
$\newcommand{\Lyb}{Ly\rm{\beta}}$
$\newcommand{\fescLya}{f_{\rm{esc}}^{\rm{Ly\alpha}}}$
$\newcommand{\fescLyC}{f_{\rm{esc}}^{\rm{LyC}}}$
$\newcommand{\Hg}{H\rm{\gamma}}$
$\newcommand{\Hb}{H\rm{\beta}}$
$\newcommand{\Ha}{H\rm{\alpha}}$
$\newcommand{\OIII}{[\ion{O}{III}]}$
$\newcommand{\xiion}{\xi_{ion}}$
$\newcommand{\jwst}{{\it JWST}}$
$\newcommand{\cw}[1]{\textcolor{red}{[#1]}}$</div>



<div id="title">

# Quasi-hostless LRDs with intrinsically red, faint UV continua

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.28209-b31b1b.svg)](https://arxiv.org/abs/2611.28209)<mark>Appeared on: 2026-11-24</mark> -  _15 pages, 8 figures, 1 table. Submitted to A&A. Comments are welcome_

</div>
<div id="authors">

C. Witten, et al. -- incl., <mark>A. d. Graaff</mark>

</div>
<div id="abstract">

**Abstract:** High-redshift objects characterised by very weak UV relative to their strong, red rest-optical continua, with broad Balmer lines, appear to represent little red dots (LRDs) in their purest form: sources dominated by emission from their central engine (CE), sometimes dubbed black hole stars (BH*s). Previous work has selected BH*-dominated objects, where a BH* template dominates the rest-optical emission; this remains agnostic to the rest-UV. We therefore select low FUV-to-optical continuum ratio ( $L_{\rm 1500}/L_{5100}<0.03$ ) objects resembling MoM-BH* and The Cliff, from a spectroscopic sample of 132 LRDs at $2\lesssim z\lesssim9$ . This identifies three LRDs that we newly classify as BH*-like objects, more than doubling the existing sample: two at $z>6$ and one at $z\sim3$ . The UV slope reddens as the NUV- ( $L_{\rm 3500}$ ) and FUV-to-optical ratios fall, with $\beta_{\rm UV}$ rising from $-1.8$ at $L_{\rm 1500}/L_{5100}\sim 0.5$ to $-0.5$ at $L_{\rm 1500}/L_{5100}<0.03$ , beyond values typically seen at these redshifts. As the UV-to-optical ratio weakens, gas-enshrouded BH emission appears to dominate, leading to the quasi-hostless regime: a weak, red UV continuum emerges alongside extreme Balmer decrements and weak forbidden line emission. This reddening could be reproduced by a simple model of a blackbody and hydrogen nebular emission leaking through a high column density screen of neutral hydrogen in the $n=2$ level ( $N_{\rm H, n=2}\approx10^{17} {\rm cm^{-2}}$ ), combined with a relatively weak host galaxy, although alternative CE scenarios may also reproduce this reddening. The host galaxy's growing FUV contribution in this model results in a testable UV morphological evolution with wavelength. This work identifies a family of quasi-hostless LRDs that share a common, intrinsically red, rest-UV continuum originating from their central engine.

</div>

<div id="div_fig1">

<img src="tmp_2611.28209/./figures/3micron_vs_UV_slopes_vertical.png" alt="Fig1" width="100%"/>

**Figure 1. -** (Top:) Histogram of the FUV-to-optical continuum ratio, $L_{1500}/L_{5100}$, of LRDs  ([de Graaff, et. al 2025](https://ui.adsabs.harvard.edu/abs/2025arXiv251121820D))  and BH*-dominated objects  ([Weibel, et. al 2026](https://ui.adsabs.harvard.edu/abs/2026arXiv260617271W)) , at $z<6$(grey) and $z>6$(blue). (Middle:) Distribution of the NUV-to-optical continuum ratio, $L_{3500}/L_{5100}$, as a function of the FUV-to-optical continuum ratio. (Bottom:) Distribution of the UV-continuum slope as a function of the FUV-to-optical continuum ratio. The pink shaded regions indicate the distribution of 75\%(dark) and 90\%(light) of galaxies from the DJA at $z>4$. Filled circular points indicate LRDs from [de Graaff, et. al (2025)](https://ui.adsabs.harvard.edu/abs/2025arXiv251121820D), filled diamonds show BH*-dominated objects from [Weibel, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026arXiv260617271W), and red unfilled diamonds show the values taken from the FUV-to-optical stacks in Section \ref{sec:red_slope}. Previously identified BH*s, MoM-BH* (blue) and The Cliff (grey), are indicated with stars, while the grey shaded region at $\log_{10}(L_{1500}/L_{5100})<-1.5$ indicates our criteria for selecting quasi-hostless LRDs. The FUV- and NUV-to-optical ratio of BH*-like objects are beyond the 96th percentile of the LRD and BH* dominated UV-to-optical continuum ratio distribution, and with this decreasing continuum ratio emerges a red UV-continuum slope. (*fig:UV-to-opt*)

</div>
<div id="div_fig2">

<img src="tmp_2611.28209/./figures/stacked_spectra.png" alt="Fig6" width="100%"/>

**Figure 6. -** (Top:) Median stacked spectra of LRDs based on their FUV-to-optical continuum ratio. These are split into roughly equal bins in continuum ratio, $L_{1500}/L_{5100}:[-1.8,  -1,  -0.75,  -0.5,  0.3]$. (Bottom:) The objects are instead stacked based on their NUV-to-optical continuum ratio, in bins of $L_{3500}/L_{5100}:[-1.4,  -0.75,  -0.5,  -0.3,  0.2]$. The spectra are coloured based on the median UV-to-optical ratio of the galaxies within each stacked spectrum. The best-fit UV-continuum of each stack is indicated with a grey solid line, and their UV-continuum slopes are indicated in the top left of each panel. Regardless of whether the UV continuum is measured in the FUV or NUV, we see an evolution in the UV slope and the weakening of forbidden line emission with decreasing UV-to-optical continuum ratio. (*fig:UV-opt-stacks*)

</div>
<div id="div_fig3">

<img src="tmp_2611.28209/./figures/emission_lines.png" alt="Fig7" width="100%"/>

**Figure 7. -** The evolution of the emission line properties of LRDs and BH*-dominated objects as a function of the FUV-to-optical (left) and NUV-to-optical (right) continuum ratio. (Top:) The $\OIII$$\lambda 5008$ to $\Ha$ emission line flux ratio, (Middle:) the $\Ha$/$\Hb$ emission line flux ratio (Balmer decrement), and (Bottom:) the equivalent width of the $\OIII$ emission line (EW$_{$\OIII$}$) are shown as a function of the UV-to-optical continuum ratio. The same markers are used as in Figure \ref{fig:UV-to-opt}, except where black downward pointing triangles indicate upper bounds. The grey dashed line in the middle row indicates the Balmer decrement assuming Case B recombination  ([Osterbrock and Ferland 2006]()) . The emission line properties of LRDs and BH*-dominated objects evolve as a function of the UV-to-optical continuum ratio, seemingly tracing the reduction in the contribution of a host galaxy: reducing the forbidden line emission and facilitating a departure from the Case B recombination assumption. (*fig:emission_lines*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.28209"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\zh}[1]{\begin{CJK*}{UTF8}{gbsn}#1\end{CJK*}}$
$\newcommand\arraystretch{1.5}$
$\newcommand\arraystretch{1.5}$</div>



<div id="title">

# Discovery of Two Hyperluminous Infrared Galaxies at $z\sim6$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.29560-b31b1b.svg)](https://arxiv.org/abs/2611.29560)<mark>Appeared on: 2026-11-24</mark> -  _Submitted to A&A_

</div>
<div id="authors">

S. Jin, et al. -- incl., <mark>E. Schinnerer</mark>

</div>
<div id="abstract">

**Abstract:** Hyperluminous infrared galaxies (HyLIRGs) at cosmic dawn are key to understanding the formation of the first massive galaxies. We report the spectroscopic confirmation of two unlensed HyLIRGs at $z\sim6$ in the COSMOS field, AzTECC63 and S2COS-84, originally selected from single-dish AzTEC and SCUBA-2 surveys. Using ALMA and NOEMA observations, we robustly detected CO and [ C ${\sc ii}$ ] lines in both galaxies and unambiguously confirmed their redshifts to be $z=6.092$ and $z=5.90$ , respectively. Panchromatic SED fitting reveals dust-obscured star formation rates of $\approx1000$ and $\approx2300 M_\odot{\rm yr}^{-1}$ , ranking them among the most intensely star-forming systems known at $z \gtrsim 6$ . High-resolution ALMA observations reveal an extremely compact morphology for S2COS-84, with a dust-continuum effective radius of $\approx0.3$ kpc, implying a star-formation rate surface density above the Eddington limit. The far-infrared SEDs of both galaxies, together with the compact size of S2COS-84, indicate warm dust ( $T_{\rm dust} \approx 80$ K) that is likely optically thick at far-infrared wavelengths. Both galaxies exhibit an extreme [ C ${\sc ii}$ ] deficit, with $L_{\rm[C II]}/L_{\rm IR} \approx 10^{-3.7}$ , an order of magnitude below that of normal star-forming galaxies. Their molecular gas masses, derived consistently from both [ C ${\sc ii}$ ] and dust, together with their IR-derived SFRs imply short gas depletion times of $\approx40$ --60 Myr, making them plausible progenitors of the earliest massive quiescent galaxies found at $z \sim 5$ .

</div>

<div id="div_fig1">

<img src="tmp_2611.29560/./figure1.png" alt="Fig6" width="100%"/>

**Figure 6. -** ALMA and NOEMA spectra and images of AzTECC63 at $z=6.092$. The 3 mm spectrum is shown at the top and the 1 mm spectrum at the bottom left. Detected lines are highlighted in green and labeled. Gray areas mark the gaps between spectral windows. The red line shows the fit to the continuum. The bottom-right panels present the cleaned continuum and line images, with the beam shown in the corner. Contours start at $3\sigma$ and increase in steps of $2\sigma$. (*fig1*)

</div>
<div id="div_fig2">

<img src="tmp_2611.29560/./figure2.png" alt="Fig7" width="100%"/>

**Figure 7. -** Same as Fig. \ref{fig1}, but for S2COS-84 at $z=5.90$. In bottom-left panel, the gray area marks the edge of ALMA Band 7. Only the low-resolution ALMA 1 mm continuum and [C {\sc ii}] data are shown here. (*fig2*)

</div>
<div id="div_fig3">

<img src="tmp_2611.29560/./LCII-LIR.png" alt="Fig1" width="100%"/>

**Figure 1. -** $L_{\rm[CII]}/L_{\rm IR}$ versus $L_{\rm IR}$ for the two galaxies of this work and for literature samples. The $z>6$ sources are from \cite{Matthee2019CII}, with $L_{\rm IR}$ estimated by \cite{Schaerer2020Cii} assuming $T_{\rm dust}=45$ K. The ALPINE sample is from \cite{Schaerer2020Cii}. The $z>3$ and $z<3$ samples are from \cite{Gullberg2015Cii} and \cite{Zanella2018Cii}, respectively.
 (*fig-CII-IR*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.29560"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\Gaia}{\textit{Gaia}}$
$\newcommand{\SPHEREx}{\textit{SPHEREx}}$
$\newcommand{\teff}{\ensuremath{T_{\mathrm{eff}}}}$
$\newcommand{\mh}{\ensuremath{[\mathrm{M/H}]}}$
$\newcommand{\feh}{\ensuremath{[\mathrm{Fe/H}]}}$
$\newcommand{\Msun}{\ensuremath{M_\odot}}$
$\newcommand{\MKs}{\ensuremath{M_{K_s}}}$
$\newcommand{\GKs}{\ensuremath{G-K_s}}$
$\newcommand{\dchi}{\ensuremath{\Delta\chi^2}}$
$\newcommand{\mum}{\ensuremath{ \mu\mathrm m}}$
$\newcommand{\dbltopfraction}{0.9}$
$\newcommand{\dblfloatpagefraction}{0.85}$
$\newcommand{\textfraction}{0.1}$</div>



<div id="title">

# Unresolved binaries from $\Gaia$ XP to $\SPHEREx$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.29702-b31b1b.svg)](https://arxiv.org/abs/2611.29702)<mark>Appeared on: 2026-11-24</mark> -  _24 pages, 16 figures, 11 tables_

</div>
<div id="authors">

<mark>J. Li</mark>, et al. -- incl., <mark>H.-W. Rix</mark>, <mark>J. Müller-Horn</mark>, <mark>K. El-Badry</mark>, <mark>B. Pennell</mark>

</div>
<div id="abstract">

**Abstract:** Unresolved companions alter the luminosity and spectra of a star, but a colour--magnitude diagram (CMD) reveals them only when the system is over-luminous, and a $\Gaia$ photocentre orbit constrains only a combination of the mass and flux ratios. We fit the absolute $\Gaia$ BP/RP (XP) and $\SPHEREx$ spectra (XPS) of about 71 000 low-mass stars in the $\Gaia$ Catalogue of Nearby Stars with the J-CaPS XPS emulator, which predicts single-star and coeval-binary spectra on an absolute flux scale.First, more than half of known binaries lie within the main-sequence (MS) band of the CMD, because the companion is faint or lifts a metal-poor primary onto the single-star ridge. XPS recovers 86 per cent of the binaries that a CMD cut finds and 44 per cent of those it misses.Second, a small photocentre motion can come from a faint companion or a luminous one of nearly equal mass. XPS distinguishes them and recovers the mass ratios of double-lined binaries. Among substellar candidates with radial-velocity follow-up, it ranks all 11 binaries above all 16 confirmed brown-dwarf or planet hosts. The public package \texttt{spbin} provides this check before follow-up.Third, where XPS detects no companion light, we place an upper limit on the light of an MS companion. Among $\Gaia$ DR3 astrometric orbits with clean spectral fits, 2841 require more companion mass than XPS allows for a single MS star. Within 200 pc, their companion masses cluster at those of typical white dwarfs, and nearly circular orbits support this interpretation.Of 24 companions that joint radial-velocity and astrometric orbits show to be faint, XPS recovers 22, compared with 14 selected by the absence of a colour excess. Applied to forthcoming $\Gaia$ DR4 orbital solutions, XPS can help distinguish luminous binaries from substellar companions and identify candidate compact companions whose dynamical masses exceed the limits allowed by their light.

</div>

<div id="div_fig1">

<img src="tmp_2611.29702/./figures/fig_dark_companions.png" alt="Fig8" width="100%"/>

**Figure 8. -** Companions too faint for their mass among the $\Gaia$ DR3 astrometric orbits within 200 pc. (a) The criterion: the dark-companion mass $M_{2,\rm dark}$, the lowest mass that the orbit allows, against $M_{2,\rm up}$, the largest MS companion mass that XPS allows. In the shaded region an MS companion of the orbital mass is brighter than XPS allows. Black: $M_{2,\rm dark}$ exceeds $M_{2,\rm up}$ by more than twice its error (MS companion excluded); orange: companion light ($D>25$); grey: neither. Literature systems at their published companion mass: RV-faint companions (purple stars; filled: Group 1, open: Group 2;  ([Yamaguchi, et. al 2024]()) ), neutron-star candidates (ochre diamonds;  ([El-Badry, et. al 2024](https://doi.org/10.33232/001c.121261)) ) and Gaia BH1 (at the right edge). (b) Dark-companion masses of classes II and III of [Shahaf, et. al (2023)]()(grey), of the orbits with the MS companion excluded (black) and of the class-II and class-III orbits with companion light (orange); ticks: the published masses of the RV-faint companions and the neutron-star candidates. (c) Cumulative distribution of the orbital eccentricity $e$ for class II by outcome, for the RV-faint companions and for the excluded class-III orbits (dashed). (*fig:dark-companions*)

</div>
<div id="div_fig2">

<img src="tmp_2611.29702/./figures/fig_outcomes.png" alt="Fig3" width="100%"/>

**Figure 3. -** Fits to three stars with RV follow-up. (a) Companion light ($D=164$; Eq. \eqref{eq:dchi2}). In each of three colour bins we selected the independently known binary within 0.15 mag of the ridge with $D$ closest to the bin median. The star shown is from the bluest bin. (b) No companion light: HIP 60865 hosts a substellar companion  ([Barbato, et. al 2026]()) . (c) MS companion excluded: for J1848$-$5709  ([Yamaguchi, et. al 2024]())  the orange model is refitted with an MS companion that satisfies the mass function of the $\Gaia$ orbit (0.57 $\Msun$). Left: CMD with the ridge (solid), upper edge of the MS (dashed) and equal-mass line (dotted). Middle: flux at 10 pc (grey), single-star model (blue) and binary model (orange). Right: data and binary model relative to the single-star model, on a common scale. Open circles average six $\SPHEREx$ channels, squares mark $JHK_s$ and diamonds the binary model in $JHK_s$. (*fig:outcomes*)

</div>
<div id="div_fig3">

<img src="tmp_2611.29702/./figures/fig5_gcns_candidates.png" alt="Fig5" width="100%"/>

**Figure 5. -** Binary candidates in the GCNS at 0.5--0.9 $\Msun$. (a) CMD; orange: detected binaries ($D>25$) on the MS (below the dashed line). (b) Fraction of MS stars with a binary signature independent of XPS, for the detected binaries ($D>25$), the same number of stars highest above the ridge, all MS stars and the stars without a detection ($D<25$). (c) Distribution of $D$ for the sample, the SB2 (green) and $\Gaia$ SB1 (orange) benchmark binaries and their controls (dashed); dotted: $D=25$. (*fig:gcns-candidates*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.29702"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\msun}{\mathrm{M}_{\odot}}$
$\newcommand{\MG}{M_{\mathrm{G}}}$
$\newcommand{\Mtot}{m_{\rm tot}}$
$\newcommand{\mh}{[\mathrm{M}/\mathrm{H}]}$
$\newcommand{\dd}{ \mathrm{d}}$
$\newcommand{\ut}{\widetilde u}$
$\newcommand{\zh}{\widehat z}$
$\newcommand{\StudentT}{\mathcal{T}}$
$\newcommand{\Normal}{\mathcal{N}}$
$\newcommand{\uunit}{\mathrm{km s^{-1} au^{1/2}}}$
$\newcommand{\cn}[1]{\begin{CJK*}{UTF8}{gbsn}#1\end{CJK*}}$
$\newcommand{\indicator}{\mathbb{I}}$
$\newcommand{\softplus}{\operatorname{softplus}}$
$\newcommand{\sigmoid}{\operatorname{sigmoid}}$
$\newcommand\@captype{table}$
$\newcommand\@captype{figure}$</div>



<div id="title">

# Dynamical calibration of a metallicity-dependent stellar mass--luminosity relation from Gaia wide binaries

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.30393-b31b1b.svg)](https://arxiv.org/abs/2611.30393)<mark>Appeared on: 2026-11-24</mark> -  _Submitted to A&A_

</div>
<div id="authors">

Y.-T. Wang, et al. -- incl., <mark>H.-W. Rix</mark>, <mark>Y.-S. Ting</mark>, <mark>J. Li</mark>

</div>
<div id="abstract">

**Abstract:** The mass of a main-sequence star is usually estimated from its luminosity through a mass--luminosity relation (MLR). This relation depends on metallicity, and its empirical calibration relies on few stars with measured masses. We calibrate the MLR of single main-sequence stars in the Gaia $G$ band as a function of metallicity with $10 559$ wide binaries. The projected relative speed and separation of a wide binary constrain its total mass statistically. We describe the stellar mass as a monotone function of absolute magnitude $\MG$ and metallicity $\mh$ and fit this function to all pairs. The metallicities come from Gaia low-resolution (XP) spectra and are calibrated by requiring that both stars of a pair have the same metallicity. The likelihood includes the measurement noise, the selection of the sample and pairs with an undetected close companion. The data constrain the MLR over $3.7\lesssim\MG\lesssim11.8$ mag and $-0.5\lesssim\mh\lesssim0.4$ on our calibrated metallicity scale, with a median statistical uncertainty of $2.5\%$ in mass. Tests of the model assumptions indicate an additional systematic uncertainty of up to $5\%$ . At solar metallicity the MLR agrees with the PARSEC models to within $5\%$ . At the faint end, the MIST, BaSTI, Dartmouth and BHAC15 models predict masses 8 to $22\%$ below our MLR. At fixed $\MG$ the mass increases with metallicity, by $12\%$ per $0.5$ dex at $\MG=5$ mag and by $26\%$ at $\MG=11$ mag. The dependence is $1\sigma$ to $2\sigma$ stronger than PARSEC predicts. Published masses from orbits, eclipsing binaries and asteroseismology lie within about $10\%$ of our MLR for $\MG<10.5$ mag. For fainter stars they are about $17\%$ lower. The origin of the difference at the faint end is not yet understood. Because a shift of $0.1$ dex in metallicity changes the mass by up to $5\%$ , MLRs should be compared on a common metallicity scale. The MLR is available as the Python package \texttt{wbmlr} ( $\protect$ $\url{https://github.com/yt-wangg/wide-binary-mlr}$ ).

</div>

<div id="div_fig1">

<img src="tmp_2611.30393/./figs/outlier_template_dist.png" alt="Fig2" width="100%"/>

**Figure 2. -** Distribution of the orbit factor $\ut$ for the good component (grey dashed line) and for the outlier component (orange lines). The outlier distributions $h_j$ are averaged over the pairs in each third of the distance distribution. All curves are normalised to unit area. (*fig:template_dist*)

</div>
<div id="div_fig2">

<img src="tmp_2611.30393/./figs/u_predictive.png" alt="Fig4" width="100%"/>

**Figure 4. -** Observed distribution of $u$ for the $10 559$ pairs (points with Poisson errors) and the posterior prediction of the model with the selection (blue line). The dashed and dotted lines are the good component and the outlier component. (*fig:upred*)

</div>
<div id="div_fig3">

<img src="tmp_2611.30393/./figs/mlr_summary.png" alt="Fig11" width="100%"/>

**Figure 11. -** The MLR at four metallicities from $Z=-0.6$ to $+0.3$. Panel (a) shows our MLR with its posterior interval (solid lines and bands) and PARSEC at the same metallicities (dashed lines). The dash-dotted line is the relation of [Hwang, et. al (2024)](https://ui.adsabs.harvard.edu/abs/2024MNRAS.528.4272H) along the ridge of the main sequence, and the star marks the solar mass anchor. Panel (b) shows masses relative to PARSEC at $Z=0$. It includes our MLR at $Z=0$ and the published masses of stars within $0.15$ dex of solar metallicity (symbols, Sect. \ref{sec:disc_masses}). (*fig:summary*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.30393"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

162  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

20  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

6  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
